In [ ]:
import os, shutil, rasterio, sys
sys.path.append('backend/app/')
from rasterio.features import shapes, rasterize
from rasterio.mask import mask
from shapely.geometry import shape
import geopandas as gpd, pandas as pd
import numpy as np
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
np.random.seed(42)

In [ ]:
def keep_polygon(geom):
    if geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 0: return None
        return polys[0]
    return geom

def fix_invalid_polygon(gdf, cols):
    gdf_new, name = gdf.copy(), cols[0]
    gdf_valid, gdf_nan = gdf_new[gdf_new[name] != ''], gdf_new[gdf_new[name] == '']
    if gdf_nan.shape[0] > 0:
        gdf_valid['geometry'] = gdf_valid['geometry'].apply(keep_polygon)
        gdf_nan['geometry'] = gdf_nan['geometry'].apply(keep_polygon)
        # Spatial join nearest
        gdf_filled = gpd.sjoin_nearest(
            gdf_nan, gdf_valid[['geometry', name]], how='left', distance_col='dist'
        )
        gdf_filled = gdf_filled.drop_duplicates(subset='_id')
        gdf_new.loc[gdf_filled.index, cols] = gdf_valid.loc[gdf_filled['index_right'], cols].values
    gdf_new['geometry'] = gdf_new['geometry'].apply(keep_polygon)
    return gdf_new

def clip_catchment(catchment, raster_path, out_path, inside=False):
    with rasterio.open(raster_path) as src:
        geoms = catchment.geometry.values
        # Clip raster
        out_image, out_transform = mask(
            src, geoms, crop=False, nodata=-9999, invert=inside
        )
        out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform, "nodata": -9999
    })
    with rasterio.open(out_path, "w", **out_meta) as dest:
        dest.write(out_image)


In [ ]:
folder = 'wflow_model'
catchment_path = r'backend\src\flow_samples\catchment.geojson'
terrain_path = r'backend\src\flow_samples\dtm10.tif'
flowacc_path = r'backend\src\flow_samples\dtm10_flowacc.tif'
soil_path = r'backend\src\flow_samples\soil.geojson'
land_path = r'backend\src\flow_samples\landcover.geojson'
river_path = r'backend\src\flow_samples\river.geojson'
weather_path = r'backend\src\flow_samples\alesund_weather.csv'
terrain = rasterio.open(terrain_path)
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)
weather = pd.read_csv(weather_path)

In [205]:
# Clip dtm to catchment
catchment_UTM = catchment.to_crs(terrain.crs)
terrain_out_path = os.path.normpath(os.path.join(folder, 'inputs', "dtm_clipped.tif"))
clip_catchment(catchment_UTM, terrain_path, terrain_out_path)

In [ ]:
river_UTM = river.to_crs(terrain.crs)
cols = {
    'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)
}
select_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in select_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[mask, col] = np.round(np.random.uniform(low, high, mask.sum()), 3)
transform = terrain.transform
minx, miny, maxx, maxy = river_UTM.total_bounds
# Create river shape
shapes = ((geom, 1) for geom in river_UTM.geometry)
raster = rasterize(
    shapes=shapes, out_shape=(terrain.height, terrain.width),
    transform=transform, fill=0, dtype="float32", all_touched=True
)
out_meta = {
    "driver": "GTiff", "height": terrain.height, "width": terrain.width,
    "count": 1, "dtype": "uint8", "crs": terrain.crs,
    "transform": transform, "nodata": 0
}
river_path = os.path.normpath(os.path.join(folder, 'inputs', "river.tif"))
with rasterio.open(river_path, "w", **out_meta) as dst:
    dst.write(raster, 1)

In [251]:
river_UTM

,id,_id,width,depth,manning_n,geometry
0,0,1,None,None,None,"LINESTRING (57664.983 6954624.961, 57645.023 6..."
1,1,2,None,None,None,"LINESTRING (57664.983 6954624.961, 57664.985 6..."
2,2,3,None,None,None,"LINESTRING (57694.997 6954715.022, 57694.972 6..."
3,3,4,None,None,None,"LINESTRING (57694.997 6954715.022, 57714.984 6..."
4,4,5,None,None,None,"LINESTRING (61224.992 6954904.957, 61224.992 6..."
...,...,...,...,...,...,...
63,63,64,None,None,None,"LINESTRING (63235.011 6957064.972, 63244.971 6..."
64,64,65,None,None,None,"LINESTRING (62975.01 6957084.993, 62995.028 69..."
65,65,66,None,None,None,"LINESTRING (63524.997 6957154.997, 63515.03 69..."
66,66,67,None,None,None,"LINESTRING (64554.979 6957544.984, 64555.009 6..."


In [253]:
river_UTM

,id,_id,width,depth,manning_n,geometry
0,0,1,0.780,1.298,0.046,"LINESTRING (57664.983 6954624.961, 57645.023 6..."
1,1,2,1.904,4.948,0.051,"LINESTRING (57664.983 6954624.961, 57664.985 6..."
2,2,3,1.477,4.089,0.041,"LINESTRING (57694.997 6954715.022, 57694.972 6..."
3,3,4,1.217,1.795,0.059,"LINESTRING (57694.997 6954715.022, 57714.984 6..."
4,4,5,0.354,1.022,0.059,"LINESTRING (61224.992 6954904.957, 61224.992 6..."
...,...,...,...,...,...,...
63,63,64,0.746,1.888,0.053,"LINESTRING (63235.011 6957064.972, 63244.971 6..."
64,64,65,0.598,1.479,0.049,"LINESTRING (62975.01 6957084.993, 62995.028 69..."
65,65,66,1.108,2.350,0.033,"LINESTRING (63524.997 6957154.997, 63515.03 69..."
66,66,67,0.325,4.772,0.035,"LINESTRING (64554.979 6957544.984, 64555.009 6..."


In [167]:
# Fix invalid soil polygon
soil_UTM = soil.to_crs(terrain.crs)
soil_cols = ['soil', 'theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = fix_invalid_polygon(soil_UTM, soil_cols)
soil_out_path = os.path.normpath(os.path.join(folder, 'inputs', "soil.geojson"))
soil_UTM.to_file(soil_out_path, driver='GeoJSON', encoding='utf-8')

In [168]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = fix_invalid_polygon(land_UTM, land_cols)
land_out_path = os.path.normpath(os.path.join(folder, 'inputs', "land.geojson"))
land_UTM.to_file(land_out_path, driver='GeoJSON', encoding='utf-8')

In [170]:
# Process river polygon
river_UTM = river.to_crs(terrain.crs)
# river_UTM['geometry'] = river_UTM['geometry'].apply(keep_polygon)
# river_out_path = os.path.normpath(os.path.join(folder, 'inputs', "river.geojson"))
# river_UTM.to_file(river_out_path, driver='GeoJSON', encoding='utf-8')
river_UTM

,id,_id,width,depth,manning_n,geometry
0,0,1,None,None,None,"LINESTRING (57664.983 6954624.961, 57645.023 6..."
1,1,2,None,None,None,"LINESTRING (57664.983 6954624.961, 57664.985 6..."
2,2,3,None,None,None,"LINESTRING (57694.997 6954715.022, 57694.972 6..."
3,3,4,None,None,None,"LINESTRING (57694.997 6954715.022, 57714.984 6..."
4,4,5,None,None,None,"LINESTRING (61224.992 6954904.957, 61224.992 6..."
...,...,...,...,...,...,...
63,63,64,None,None,None,"LINESTRING (63235.011 6957064.972, 63244.971 6..."
64,64,65,None,None,None,"LINESTRING (62975.01 6957084.993, 62995.028 69..."
65,65,66,None,None,None,"LINESTRING (63524.997 6957154.997, 63515.03 69..."
66,66,67,None,None,None,"LINESTRING (64554.979 6957544.984, 64555.009 6..."


In [ ]:
crs = terrain.crs
catchment = catchment.to_crs(crs)
soil = soil.to_crs(crs)
land = land.to_crs(crs)
river = river.to_crs(crs)


In [23]:
new.to_file('catchment.geojson', driver='GeoJSON', encoding='utf-8')

In [8]:
land[land['land'] == '']

,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,0,1,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ..."
3,3,4,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ..."
5,5,6,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37229 62.45831, 6.37229 62.45823, ..."
8,8,9,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45856, 6.37202 62.45847, ..."
11,11,12,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37082 62.45783, 6.37082 62.45775, ..."
...,...,...,...,...,...,...,...,...,...,...
8815,8815,8816,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.47595 62.48438, 6.47595 62.4843, 6..."
8819,8819,8820,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.47782 62.4847, 6.47782 62.48462, 6..."
8823,8823,8824,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.47103 62.48358, 6.47103 62.4835, 6..."
8826,8826,8827,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.47196 62.48366, 6.47196 62.48358, ..."


In [48]:
gdf

,_id,geometry,land,LAI,root_depth,interception,manning_n,albedo,kc
0,1,"POLYGON ((6.64177 62.52099, 6.64177 62.52091, ...",Dense vegetation,5.0,1.5,3.0,0.4,0.13,1.1
1,2,"POLYGON ((6.63032 62.52035, 6.63032 62.52027, ...",None,None,None,None,None,None,None
2,3,"POLYGON ((6.63045 62.52035, 6.63045 62.52027, ...",Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9
3,4,"POLYGON ((6.63099 62.52035, 6.63099 62.52027, ...",Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9
4,5,"POLYGON ((6.63112 62.52035, 6.63112 62.52027, ...",Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3
...,...,...,...,...,...,...,...,...,...
56649,56650,"POLYGON ((6.33183 62.41953, 6.33183 62.41945, ...",None,None,None,None,None,None,None
56650,56651,"POLYGON ((6.32917 62.41937, 6.32917 62.41929, ...",Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3
56651,56652,"POLYGON ((6.3297 62.41969, 6.3297 62.41961, 6....",Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9
56652,56653,"POLYGON ((6.3293 62.41937, 6.3293 62.41929, 6....",None,None,None,None,None,None,None


In [36]:
geoms

[{'geometry': <POLYGON ((6.642 62.521, 6.642 62.521, 6.642 62.521, 6.642 62.521, 6.642 62....>,
  'land': 'Dense vegetation'},
 {'geometry': <POLYGON ((6.63 62.52, 6.63 62.52, 6.63 62.52, 6.63 62.52, 6.63 62.52))>,
  'land': ''},
 {'geometry': <POLYGON ((6.63 62.52, 6.63 62.52, 6.631 62.52, 6.631 62.52, 6.63 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52))>,
  'land': 'Impervious/Urban'},
 {'geometry': <POLYGON ((6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52))>,
  'land': ''},
 {'geometry': <POLYGON ((6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.64 62.521,

In [15]:
path = r"backend\src\flow_samples\landcover.geojson"
gdf = gpd.read_file(path)

In [14]:
gdf['land'] = gdf['land'].astype(str)
gdf

,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,0,1,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ..."
1,1,2,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((6.37229 62.45823, 6.37229 62.45815, ..."
2,2,3,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37215 62.45831, 6.37215 62.45815, ..."
3,3,4,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ..."
4,4,5,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37162 62.45831, 6.37162 62.45823, ..."
...,...,...,...,...,...,...,...,...,...,...
8832,8832,8833,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.46464 62.48277, 6.46464 62.48269, ..."
8833,8833,8834,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46225 62.48301, 6.46225 62.48293, ..."
8834,8834,8835,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46424 62.4835, 6.46424 62.48342, 6..."
8835,8835,8836,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.47582 62.48494, 6.47582 62.48486, ..."


In [ ]:
gdf['land'] = np.where(gdf['land']=='', 'None', gdf['land'])

In [11]:
gdf


,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,0,1,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ..."
1,1,2,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((6.37229 62.45823, 6.37229 62.45815, ..."
2,2,3,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37215 62.45831, 6.37215 62.45815, ..."
3,3,4,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ..."
4,4,5,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37162 62.45831, 6.37162 62.45823, ..."
...,...,...,...,...,...,...,...,...,...,...
8832,8832,8833,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.46464 62.48277, 6.46464 62.48269, ..."
8833,8833,8834,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46225 62.48301, 6.46225 62.48293, ..."
8834,8834,8835,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46424 62.4835, 6.46424 62.48342, 6..."
8835,8835,8836,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.47582 62.48494, 6.47582 62.48486, ..."
